In [1]:
# Imports to enable plotting functionality
import numpy as np
import matplotlib.pyplot as plt
import bz2
import pickle
import _pickle as cPickle
import os 
from scipy.io import loadmat

## Compute rotations (Praful: you can ignore this)

In [10]:
# Define initial positions of the three electrodes
basis_elecs = np.array([[0, 0], [30, 15], [30, -15]])

# Shift electrodes by -1030 in x
x_offset = 0
basis_elecs[:, 0] += x_offset

print("Shifted electrode positions:", basis_elecs)

# Find center of the electrodes
center = np.mean(basis_elecs, axis=0)

# Translate electrodes so that the center is at the origin
translated_elecs = basis_elecs - center

# Convert 10 degrees to radians
theta = np.radians(90)

# Define the 2D rotation matrix
rotation_matrix = np.array([[np.cos(theta), -np.sin(theta)],
                            [np.sin(theta), np.cos(theta)]])

# Rotate each electrode around the center
rotated_translated_elecs = np.dot(translated_elecs, rotation_matrix.T)

# Translate back to original center
rotated_elecs = rotated_translated_elecs + center

# Round electrodes to 3 decimal places
rotated_elecs = np.round(rotated_elecs, 3)

# Print the rotated electrode positions
print("Rotated electrode positions around the center:")
for i, elec in enumerate(rotated_elecs):
    print(f"Electrode {i+1}: {elec}")

Shifted electrode positions: [[  0   0]
 [ 30  15]
 [ 30 -15]]
Rotated electrode positions around the center:
Electrode 1: [ 20. -20.]
Electrode 2: [ 5. 10.]
Electrode 3: [35. 10.]


## Plot the Data

In [2]:
# Pickle a file and then compress it into a file with extension 
def compressed_pickle(title, data):
    with bz2.BZ2File(title + '.pbz2', 'w') as f: 
        cPickle.dump(data, f)

# Load any compressed pickle file
def decompress_pickle(file):
    data = bz2.BZ2File(file+'.pbz2', 'rb')
    data = cPickle.load(data)
    return data

In [3]:
# Point to directory where sim data is stored, here doing it for rot0, but can change to rot10,rot20, etc
dir_path = '/Volumes/Lab/Users/vilkhu/data-backup/sim-data/triplet_rotations/rot0/'

# Output the electrodes files 
filename = dir_path+'electrodes.pkl'
data = decompress_pickle(filename)
print('Electrodes: ',data)

# Find data files in dir
data_files = []
for root,dirs,files in os.walk(dir_path):
    for file in files:
        if '_' in file:
            data_files.append(file)

# Next, extract data
i1 = []
i2 = []
i3 = []
spikes = []
for file in data_files:
        filename = dir_path+str(file)
        data = decompress_pickle(filename[:-5])
        
        # Output substring before '_' in file
        curr1 = round(float(file.split('_')[0]),2)
        curr2 = round(float(file.split('_')[1]),2)
        curr3 = round(float(file.split('_')[2][:-9]),2)
        # if np.abs(curr1) < 3.1 and np.abs(curr2) < 3.1:
        i1.append(curr1)
        i2.append(curr2)
        i3.append(curr3)
        spikes.append(data[0])


Electrodes:  [[-1030 0 42]
 [-1000 15 42]
 [-1000 -15 42]]


In [4]:
# save as matrix with columns i1, i2, i3, spikes
data = np.array([i1,i2,i3,spikes]).T
print(data)


[[ 0.21 -2.74 -3.58  1.  ]
 [-3.16  0.63 -0.21  1.  ]
 [ 1.89  1.05  2.32  1.  ]
 ...
 [-0.63  1.05  1.47  1.  ]
 [ 0.63 -4.   -4.    1.  ]
 [ 3.16  1.89 -0.21  1.  ]]


## Let's write some code to compute bi-electrode rotations 0 deg to 90 deg

In [4]:
import math

# Original electrode positions
elec1 = (-1030, 0)
elec2 = (-1000, 0)
center = (-1015, 0)

def rotate_point(x, y, cx, cy, angle_degrees):
    """
    Rotate a point (x, y) around a center point (cx, cy) by angle_degrees.
    """
    angle_radians = math.radians(angle_degrees)
    cos_theta = math.cos(angle_radians)
    sin_theta = math.sin(angle_radians)
    
    # Translate point back to origin
    x_shifted = x - cx
    y_shifted = y - cy
    
    # Rotate point
    x_rotated = x_shifted * cos_theta - y_shifted * sin_theta
    y_rotated = x_shifted * sin_theta + y_shifted * cos_theta
    
    # Translate point back to original center
    x_new = x_rotated + cx
    y_new = y_rotated + cy
    
    return x_new, y_new

# Compute and print positions for angles from 0 to 90 degrees in 5-degree increments
for angle in range(0, 91, 5):
    elec1_rotated = rotate_point(elec1[0], elec1[1], center[0], center[1], angle)
    elec2_rotated = rotate_point(elec2[0], elec2[1], center[0], center[1], angle)
    
    # Round positions to 2 decimal places
    elec1_rounded = (round(elec1_rotated[0], 2), round(elec1_rotated[1], 2))
    elec2_rounded = (round(elec2_rotated[0], 2), round(elec2_rotated[1], 2))
    
    print(f"Angle {angle} degrees:")
    print(f"  Electrode 1: {elec1_rounded}")
    print(f"  Electrode 2: {elec2_rounded}\n")



Angle 0 degrees:
  Electrode 1: (-1030.0, 0.0)
  Electrode 2: (-1000.0, 0.0)

Angle 5 degrees:
  Electrode 1: (-1029.94, -1.31)
  Electrode 2: (-1000.06, 1.31)

Angle 10 degrees:
  Electrode 1: (-1029.77, -2.6)
  Electrode 2: (-1000.23, 2.6)

Angle 15 degrees:
  Electrode 1: (-1029.49, -3.88)
  Electrode 2: (-1000.51, 3.88)

Angle 20 degrees:
  Electrode 1: (-1029.1, -5.13)
  Electrode 2: (-1000.9, 5.13)

Angle 25 degrees:
  Electrode 1: (-1028.59, -6.34)
  Electrode 2: (-1001.41, 6.34)

Angle 30 degrees:
  Electrode 1: (-1027.99, -7.5)
  Electrode 2: (-1002.01, 7.5)

Angle 35 degrees:
  Electrode 1: (-1027.29, -8.6)
  Electrode 2: (-1002.71, 8.6)

Angle 40 degrees:
  Electrode 1: (-1026.49, -9.64)
  Electrode 2: (-1003.51, 9.64)

Angle 45 degrees:
  Electrode 1: (-1025.61, -10.61)
  Electrode 2: (-1004.39, 10.61)

Angle 50 degrees:
  Electrode 1: (-1024.64, -11.49)
  Electrode 2: (-1005.36, 11.49)

Angle 55 degrees:
  Electrode 1: (-1023.6, -12.29)
  Electrode 2: (-1006.4, 12.29)

Ang